# RoadVision AI - Dataset Validation

#### Check missing files, image-label correspondence, empty labels, malformed annotations, invalid class IDs, and bounding-box coordinates.

In [3]:
from pathlib import Path
from collections import Counter, defaultdict
import math
import pandas as pd
import yaml

### Imports and dataset paths

In [4]:
try:
    ROOT = Path(ROOT)
except NameError:
    import kagglehub
    dataset_path = Path(kagglehub.dataset_download("ashfakyeafi/road-vehicle-images-dataset"))
    ROOT = dataset_path / "trafic_data"
    
    
SPLITS = ["train", "valid", "test"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Dataset root:", ROOT.resolve())
print("Root exists:", ROOT.exists())
if not ROOT.exists():
    raise FileNotFoundError(
        f"Dataset folder not found: {ROOT}\n"
        "Set ROOT to the directory containing train/, valid/, test/, and data_1.yaml."
    )

Dataset root: C:\Users\USER\.cache\kagglehub\datasets\ashfakyeafi\road-vehicle-images-dataset\versions\2\trafic_data
Root exists: True


### Load class configuration

In [5]:
yaml_path = ROOT / "data_1.yaml"

if yaml_path.exists():
    with yaml_path.open("r", encoding="utf-8") as file:
        config = yaml.safe_load(file) or {}
    class_names = config.get("names", [])
    nc = config.get("nc", len(class_names))
    if isinstance(class_names, dict):
        class_names = [class_names[k] for k in sorted(class_names, key=lambda x: int(x))]
    print("YAML:", yaml_path)
    print("Declared number of classes:", nc)
    print("Class names:", class_names)
    if class_names and nc != len(class_names):
        print("WARNING: nc does not match the number of class names.")
else:
    config = {}
    class_names = []
    nc = None
    print(f"WARNING: {yaml_path.name} was not found. Class-ID range checks need nc.")
    print("Set nc manually below if necessary, for example: nc = 18") 

YAML: C:\Users\USER\.cache\kagglehub\datasets\ashfakyeafi\road-vehicle-images-dataset\versions\2\trafic_data\data_1.yaml
Declared number of classes: 21
Class names: ['ambulance', 'army vehicle', 'auto rickshaw', 'bicycle', 'bus', 'car', 'garbagevan', 'human hauler', 'minibus', 'minivan', 'motorbike', 'pickup', 'policecar', 'rickshaw', 'scooter', 'suv', 'taxi', 'three wheelers -CNG-', 'truck', 'van', 'wheelbarrow']


### Check split folders and image–label correspondence

In [6]:
split_rows = []
pairing_issues = []

for split in SPLITS:
    split_dir = ROOT / split
    image_dir = split_dir / "images"
    label_dir = split_dir / "labels"

    if not split_dir.exists():
        split_rows.append({
            "split": split, "split_folder_exists": False,
            "images_folder_exists": False, "labels_folder_exists": False,
            "images": 0, "label_files": 0,
            "images_without_label": 0, "labels_without_image": 0
        })
        pairing_issues.append({"split": split, "issue": "Missing split folder", "file": str(split_dir)})
        continue

    for folder, label in [(image_dir, "images"), (label_dir, "labels")]:
        if not folder.exists():
            pairing_issues.append({"split": split, "issue": f"Missing {label} folder", "file": str(folder)})

    images = {
        p.stem.lower(): p for p in image_dir.iterdir()
        if image_dir.exists() and p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    }
    labels = {
        p.stem.lower(): p for p in label_dir.glob("*.txt")
    } if label_dir.exists() else {}

    images_without_label = sorted(set(images) - set(labels))
    labels_without_image = sorted(set(labels) - set(images))

    for stem in images_without_label:
        pairing_issues.append({"split": split, "issue": "Image without label file", "file": str(images[stem])})
    for stem in labels_without_image:
        pairing_issues.append({"split": split, "issue": "Label without image", "file": str(labels[stem])})

    split_rows.append({
        "split": split,
        "split_folder_exists": True,
        "images_folder_exists": image_dir.exists(),
        "labels_folder_exists": label_dir.exists(),
        "images": len(images),
        "label_files": len(labels),
        "images_without_label": len(images_without_label),
        "labels_without_image": len(labels_without_image)
    })

split_summary = pd.DataFrame(split_rows)
display(split_summary)

pairing_issues_df = pd.DataFrame(pairing_issues, columns=["split", "issue", "file"])
print(f"Pairing/folder issues found: {len(pairing_issues_df)}")
display(pairing_issues_df.head(30))

,split,split_folder_exists,images_folder_exists,labels_folder_exists,images,label_files,images_without_label,labels_without_image
0,train,True,True,True,2704,2704,0,0
1,valid,True,True,True,300,300,0,0
2,test,False,False,False,0,0,0,0


Pairing/folder issues found: 1


,split,issue,file
0,test,Missing split folder,C:\Users\USER\.cache\kagglehub\datasets\ashfak...


### Validate every annotation line

For each non-empty line, this section checks:
- Exactly five values are present
- class_id is an integer
- All four box values are numeric and finite
- Class ID is in the valid range [0, nc - 1] when nc is known
- x_center and y_center are in [0, 1]
- width and height are in (0, 1]
- The box edges remain within the image bounds

In [7]:
annotation_issues = []
label_summary = []
class_counts = Counter()
total_boxes = 0
empty_label_files = 0
labels_checked = 0

for split in SPLITS:
    label_dir = ROOT / split / "labels"
    if not label_dir.exists():
        continue

    for label_path in sorted(label_dir.glob("*.txt")):
        labels_checked += 1
        text = label_path.read_text(encoding="utf-8", errors="replace")
        lines = text.splitlines()
        nonempty_lines = [(line_no, line.strip()) for line_no, line in enumerate(lines, start=1) if line.strip()]

        if not nonempty_lines:
            empty_label_files += 1
            annotation_issues.append({
                "split": split, "file": str(label_path), "line": None,
                "issue": "Empty label file", "details": "No annotations in file"
            })
            continue

        for line_no, line in nonempty_lines:
            parts = line.split()

            if len(parts) != 5:
                annotation_issues.append({
                    "split": split, "file": str(label_path), "line": line_no,
                    "issue": "Malformed annotation", "details": f"Expected 5 values, found {len(parts)}"
                })
                continue

            # Validate class ID independently so malformed numeric values are reported clearly.
            try:
                class_float = float(parts[0])
                if not math.isfinite(class_float) or not class_float.is_integer():
                    raise ValueError
                class_id = int(class_float)
            except (ValueError, OverflowError):
                annotation_issues.append({
                    "split": split, "file": str(label_path), "line": line_no,
                    "issue": "Invalid class ID format", "details": repr(parts[0])
                })
                continue

            try:
                x_center, y_center, width, height = map(float, parts[1:])
            except ValueError:
                annotation_issues.append({
                    "split": split, "file": str(label_path), "line": line_no,
                    "issue": "Non-numeric bounding-box value", "details": " ".join(parts[1:])
                })
                continue

            if not all(math.isfinite(v) for v in [x_center, y_center, width, height]):
                annotation_issues.append({
                    "split": split, "file": str(label_path), "line": line_no,
                    "issue": "Non-finite bounding-box value", "details": "Coordinates contain NaN or infinity"
                })
                continue

            # Class ID range
            if nc is not None and not (0 <= class_id < int(nc)):
                annotation_issues.append({
                    "split": split, "file": str(label_path), "line": line_no,
                    "issue": "Invalid class ID", "details": f"class_id={class_id}; valid range is 0..{int(nc)-1}"
                })

            # Normalized YOLO coordinates
            if not (0 <= x_center <= 1 and 0 <= y_center <= 1):
                annotation_issues.append({
                    "split": split, "file": str(label_path), "line": line_no,
                    "issue": "Center outside normalized range",
                    "details": f"x_center={x_center}, y_center={y_center}"
                })

            if not (0 < width <= 1 and 0 < height <= 1):
                annotation_issues.append({
                    "split": split, "file": str(label_path), "line": line_no,
                    "issue": "Invalid width/height",
                    "details": f"width={width}, height={height}; expected 0 < width,height <= 1"
                })

            # Bounding box must fit inside the image boundaries.
            x_min, x_max = x_center - width / 2, x_center + width / 2
            y_min, y_max = y_center - height / 2, y_center + height / 2
            if not (0 <= x_min and x_max <= 1 and 0 <= y_min and y_max <= 1):
                annotation_issues.append({
                    "split": split, "file": str(label_path), "line": line_no,
                    "issue": "Bounding box extends beyond image",
                    "details": f"x=[{x_min:.4f}, {x_max:.4f}], y=[{y_min:.4f}, {y_max:.4f}]"
                })

            class_counts[class_id] += 1
            total_boxes += 1

        label_summary.append({
            "split": split,
            "label_file": label_path.name,
            "annotations": len(nonempty_lines)
        })

issues_df = pd.DataFrame(
    annotation_issues,
    columns=["split", "file", "line", "issue", "details"]
)

print("Label files checked:", labels_checked)
print("Empty label files:", empty_label_files)
print("Annotation lines / boxes parsed:", total_boxes)
print("Annotation issues (including empty files):", len(issues_df))
display(issues_df.head(50))

Label files checked: 3004
Empty label files: 2
Annotation lines / boxes parsed: 24348
Annotation issues (including empty files): 26


,split,file,line,issue,details
0,train,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,NaN,Empty label file,No annotations in file
1,train,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,NaN,Empty label file,No annotations in file
2,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,1.0,Invalid width/height,"width=0.0, height=0.0; expected 0 < width,heig..."
3,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,2.0,Invalid width/height,"width=0.0, height=0.0; expected 0 < width,heig..."
4,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,3.0,Invalid width/height,"width=0.0, height=0.0; expected 0 < width,heig..."
5,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,4.0,Invalid width/height,"width=0.0, height=0.0; expected 0 < width,heig..."
6,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,5.0,Invalid width/height,"width=0.0, height=0.0; expected 0 < width,heig..."
7,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,6.0,Invalid width/height,"width=0.0, height=0.0; expected 0 < width,heig..."
8,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,7.0,Invalid width/height,"width=0.0, height=0.0; expected 0 < width,heig..."
9,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,8.0,Invalid width/height,"width=0.0, height=0.0; expected 0 < width,heig..."


### Class distribution

In [8]:
if class_names:
    class_rows = [
        {"class_id": i, "class_name": str(name), "bounding_boxes": class_counts.get(i, 0)}
        for i, name in enumerate(class_names)
    ]
else:
    class_rows = [
        {"class_id": i, "class_name": "unknown (YAML unavailable)", "bounding_boxes": count}
        for i, count in sorted(class_counts.items())
    ]

class_distribution = pd.DataFrame(class_rows)
display(class_distribution)

print("Total parsed bounding boxes:", total_boxes)

,class_id,class_name,bounding_boxes
0,0,ambulance,70
1,1,army vehicle,43
2,2,auto rickshaw,372
3,3,bicycle,459
4,4,bus,3339
5,5,car,5474
6,6,garbagevan,3
7,7,human hauler,169
8,8,minibus,95
9,9,minivan,932


Total parsed bounding boxes: 24348


### Validation summary and export

In [9]:
issue_counts = (
    issues_df.groupby(["split", "issue"]).size().unstack(fill_value=0)
    if not issues_df.empty else pd.DataFrame()
)

print("=== DATASET VALIDATION SUMMARY ===")
print("Split folders / image-label pairing issues:", len(pairing_issues_df))
print("Label files checked:", labels_checked)
print("Empty label files:", empty_label_files)
print("Parsed bounding boxes:", total_boxes)

if not issues_df.empty:
    print("\nIssues by type:")
    display(issues_df["issue"].value_counts().rename_axis("issue").reset_index(name="count"))
else:
    print("No annotation issues found.")

# Save reports beside the dataset root.
report_dir = ROOT / "validation_reports"
report_dir.mkdir(exist_ok=True)

split_summary.to_csv(report_dir / "split_pairing_summary.csv", index=False)
pairing_issues_df.to_csv(report_dir / "pairing_issues.csv", index=False)
issues_df.to_csv(report_dir / "annotation_issues.csv", index=False)
class_distribution.to_csv(report_dir / "class_distribution.csv", index=False)

print("\nReports saved to:", report_dir.resolve())
print("- split_pairing_summary.csv")
print("- pairing_issues.csv")
print("- annotation_issues.csv")
print("- class_distribution.csv")

=== DATASET VALIDATION SUMMARY ===
Split folders / image-label pairing issues: 1
Label files checked: 3004
Empty label files: 2
Parsed bounding boxes: 24348

Issues by type:


,issue,count
0,Invalid width/height,24
1,Empty label file,2



Reports saved to: C:\Users\USER\.cache\kagglehub\datasets\ashfakyeafi\road-vehicle-images-dataset\versions\2\trafic_data\validation_reports
- split_pairing_summary.csv
- pairing_issues.csv
- annotation_issues.csv
- class_distribution.csv
